# Ex1: Models
## Name: Shaima Al-Qahtani
##Task 1: Text Generation Temperature
- Setup an account at OpenRouter.ai
- Set the OPENROUTER_API_KEY
- Select an LLM a free model like: "nvidia/nemotron-3-nano-30b-a3b:free"
- make a high-temperature model and a low-temperature one
- Ask both of them the same question and observe the answer
- Include a stylistic instruction like “make it short” or “say it backwards” or “be poetic in your response” or “use only emojis”
- Ask the model to format their answer as JSON



In [16]:
%pip install -q langchain langchain_openai langchain_community

In [17]:
import os

In [18]:
from google.colab import userdata

os.environ['OPENROUTER_API_KEY'] =  userdata.get('OPENROUTER_API_KEY')

In [19]:
from langchain_openai import ChatOpenAI
# Low Temperature (0)
model_nemotron3_nano_precise = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    # OpenRouter instead of the default OpenAI API
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

# High Temperature (0.9)
model_nemotron3_nano_creative = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0.9,
    # OpenRouter instead of the default OpenAI API
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)


In [20]:

prompt =(
    "Explain what the moon does at night."
    "What is the purpose of drinking water every day?"
    "Describe a cat in a funny way."
)

In [21]:
from pprint import pprint

print("=== Low temperature response ===")
low_response = model_nemotron3_nano_precise.invoke(prompt)
print(low_response.content)

print("\n\n=== High temperature response ===")
high_response = model_nemotron3_nano_creative.invoke(prompt)
print(high_response.content)

=== Low temperature response ===
**1. What the Moon Does at Night**

- **Reflects Sunlight:** The Moon doesn’t produce its own light. At night it shines because it reflects sunlight that hits its surface. The amount of reflected light we see changes as the Moon orbits Earth, giving us the familiar phases (new, crescent, quarter, gibbous, full).

- **Creates Tides:** The Moon’s gravity pulls on Earth’s oceans, causing the rise and fall of tides. The side of Earth closest to the Moon experiences a bulge (high tide), and the opposite side experiences another bulge due to inertia. As Earth rotates, different coastlines pass through these bulges, producing two high and two low tides each day.

- **Influences Nighttime Visibility:** By reflecting sunlight, the Moon provides natural illumination that can range from a faint glow (waning crescent) to a bright, almost daylight‑like shine (full Moon). This affects how much artificial lighting we need and even influences animal behavior and human 

In [22]:
prompt1 = 'Return this answer in JSON: {"question": "What is the moon?", "instruction": "short answer"}'

prompt2 = 'Return this answer in JSON: {"question": "Why do we drink water?", "instruction": "one sentence"}'

prompt3 = 'Return this answer in JSON: {"question": "Describe a cat", "instruction": "very short"}'

print("=== Low Temperature Responses ===")
print("Prompt 1:", model_nemotron3_nano_precise.invoke(prompt1).content)
print("Prompt 2:", model_nemotron3_nano_precise.invoke(prompt2).content)
print("Prompt 3:", model_nemotron3_nano_precise.invoke(prompt3).content)

print("\n=== High Temperature Responses ===")
print("Prompt 1:", model_nemotron3_nano_creative.invoke(prompt1).content)
print("Prompt 2:", model_nemotron3_nano_creative.invoke(prompt2).content)
print("Prompt 3:", model_nemotron3_nano_creative.invoke(prompt3).content)

=== Low Temperature Responses ===
Prompt 1: {
"question": "What is the moon?",
  "answer": "Earth's natural satellite."
}
Prompt 2: {
  "question": "Why do we drink water?",
  "instruction": "We drink water to stay hydrated, support vital bodily functions, and maintain overall health."
}
Prompt 3: {
  "question": "Describe a cat",
  "instruction": "very short"
}

=== High Temperature Responses ===
Prompt 1: {
  "answer": "The Moon is Earth's natural satellite."
}
Prompt 2: {
 "question": "Why do we drink water?",
  "instruction": "We drink water to keep our bodies hydrated, support vital physiological functions, and regulate temperature."
}
Prompt 3: {
  "answer": "Furry"
}


----------


# Task 2: Sentiment Analysis
Step 1. Restrict the model to output exactly one of: .["positive", "neutral", "negative"]

Hint: Use with Pydantic’s BaseModel subclass..with_structured_output()

Step 2. Give the model a model few sentences, and observe it’s output:

# Positive
"Kindness creates lasting joy."
"Success rewards persistent effort."
"I love Sunlight. It warms the skin."

# Negative
"Pessemestic all the time."
"The storm caused damage!"

# Neutral
"The clock ticks steadily."

In [8]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# Step 1
class SentimentOutput(BaseModel):
    sentiment: Literal["positive", "neutral", "negative"] = Field(
        ...,
        description="The sentiment of the text"
    )

structured_llm = model_nemotron3_nano_precise.with_structured_output(SentimentOutput)

# Step 2
sentences = [
    "Kindness creates lasting joy.",
    "Success rewards persistent effort.",
    "I love Sunlight. It warms the skin.",
    "Pessemestic all the time.",
    "The storm caused damage!",
    "The clock ticks steadily."
]

# Using .invoke()
print("--- Sentiment Analysis Results ---")
for text in sentences:

    result = structured_llm.invoke(text)
    print(f"Text: {text: <35} | Output: {result.sentiment}")

--- Sentiment Analysis Results ---
Text: Kindness creates lasting joy.       | Output: positive
Text: Success rewards persistent effort.  | Output: positive
Text: I love Sunlight. It warms the skin. | Output: positive
Text: Pessemestic all the time.           | Output: positive
Text: The storm caused damage!            | Output: neutral
Text: The clock ticks steadily.           | Output: neutral


-----------

# Task 3: Categorization
Restrict the model to give tags (multi-output): .["cars", "shopping", "sports", "study", "work"]

try:

"That restoration looks incredible; you have a real talent for mechanics."
"I found the perfect gift today! The staff was incredibly helpful."
"Great game today! Your teamwork was the key to that victory."
"Learning together helped me finally grasp these concepts. Thank you!"

In [9]:
from pydantic import BaseModel, Field
from typing import Literal, List

class Categorization(BaseModel):
    category: Literal["cars", "shopping", "sports", "study", "work"] = Field(
        description="The category that best describes the input text."
    )


model_with_structure = model_nemotron3_nano_precise.with_structured_output(Categorization)

In [10]:
model_with_structure.batch([
     "That restoration looks incredible; you have a real talent for mechanics.",
     "I found the perfect gift today! The staff was incredibly helpful.",
     "Great game today! Your teamwork was the key to that victory.",
     "Learning together helped me finally grasp these concepts. Thank you!",
 ])

[Categorization(category='cars'),
 Categorization(category='shopping'),
 Categorization(category='sports'),
 Categorization(category='cars')]

--------------------


# Task 4: Data Extraction
Use to have a model read a resume/CV (text file) and identify key data points such as: Candidate Name, Skills, Experience, ..etc..with_structured_output()

In [11]:
from typing import List, Optional
from pydantic import BaseModel, Field

class ResumeData(BaseModel):
    name: str = Field(description="The full name of the candidate")
    skills: List[str] = Field(description="A list of technical and soft skills")
    years_of_experience: int = Field(description="Total years of professional experience as a number")
    top_education: str = Field(description="The highest degree or university mentioned")

In [12]:
import time

resume_text = """
Shaima Al-Qahtani

Professional Summary:
An ambitious and driven individual with a deep passion for the ever-evolving world of technology.
I am highly interested in exploring cutting-edge tech trends and integrating AI solutions
into daily life.

Education:
Bachelor of Computer Science

Skills:
- Python
- SQL
- Java
"""

# تهيئة النموذج
extractor = model_nemotron3_nano_precise.with_structured_output(ResumeData)

print("--- Extracting Data ---")
try:

    extracted_info = extractor.invoke(resume_text)


    print(f"Name: {extracted_info.name}")
    print(f"Skills: {', '.join(extracted_info.skills)}")
    print(f"Experience: {extracted_info.years_of_experience} years")
    print(f"Education: {extracted_info.top_education}")

except Exception as e:
    print(f"Error: {e}")

--- Extracting Data ---
Name: Shaima Al-Qahtani
Skills: Python, SQL, Java
Experience: 0 years
Education: Bachelor of Computer Science


--------------

# Task 5: Tools
Step 1. Define 4 Python functions: , , and .add(a, b)subtract(a, b)multiply(a, b)divide(a, b)

Step 2. Copy those functions and place them in one multi-line string. like so:

system_prompt = """
You are a math assistant. You have access to these tools:

def add(a, b):
    ...
...

Return the tool name and arguments required to solve the user's request.
"""

Step 3. Define the following pydanic class:

class ToolCall(BaseModel):
    tool_name: str = Field(description="The name of the function to use")
    arguments: Dict[str, Any] = Field(description="The parameters to pass")

Step 4. Use structured output to ask the model to answer any mathematical question like: "what is two plus 5"

Observe: can you see what this could be used for? Can you extend this idea further?

In [13]:
import os
from typing import Dict, Any
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

# تعريف الدوال
def add(a, b): return a + b
def subtract(a, b): return a - b
def multiply(a, b): return a * b
def divide(a, b): return a / b if b != 0 else "Error: Division by zero"


system_prompt = """
You are a math assistant. You have access to these tools:

def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

def multiply(a, b):
    return a * b

def divide(a, b):
    return a / b

Return the tool name and arguments required to solve the user's request in JSON format.
"""

In [14]:
class ToolCall(BaseModel):
    tool_name: str = Field(description="The name of the function to use (add, subtract, multiply, or divide)")
    arguments: Dict[str, Any] = Field(description="The parameters to pass to the function, e.g., {'a': 5, 'b': 3}")

In [15]:
# تهيئة النموذج
model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY")
)

# ToolCall
chain = model_nemotron3_nano_precise.with_structured_output(ToolCall)

# السؤال الرياضي
user_question = "what is two plus 5"

try:

    result = chain.invoke([
        ("system", system_prompt),
        ("user", user_question)
    ])

    print(f"Tool to use: {result.tool_name}")
    print(f"Arguments: {result.arguments}")


    if result.tool_name == "add":
        final_answer = add(**result.arguments)
        print(f"Final Answer: {final_answer}")

except Exception as e:
    print(f"Error: {e}")

Tool to use: add
Arguments: {'a': 2, 'b': 5}
Final Answer: 7
